# LeetCode 1099 — Two Sum Less Than K bằng Python

**Link đề:** https://leetcode.com/problems/two-sum-less-than-k/
**Độ khó:** Easy · **Chủ đề:** Array, Two Pointers, Binary Search, Sorting

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def twoSumLessThanK(self, nums: List[int], k: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu đúng yêu cầu: tìm **tổng lớn nhất** của 2 phần tử khác vị trí mà tổng **nhỏ hơn hẳn** `k`; không có thì trả `-1`.
2. Viết được lời giải Brute Force `O(n²)` và chỉ ra nút thắt.
3. Học kỹ thuật **sắp xếp + hai con trỏ** (two pointers) và hiểu *vì sao* bỏ qua được nhiều cặp mà không mất đáp án.
4. Biết thêm 2 cách: **sắp xếp + tìm nhị phân** (`bisect`) và **đếm (counting sort)** tận dụng giá trị nhỏ.
5. Trace từng bước, chứng minh tính đúng, phân tích Time/Space, tự giải các biến thể.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — tổng hai số lớn nhất nhưng phải nhỏ hơn K

Cho mảng số nguyên `nums` và số nguyên `k`. Hãy trả về **giá trị lớn nhất** của `sum = nums[i] + nums[j]` với `i < j` sao cho `sum < k`. Nếu không tồn tại cặp `i, j` nào thỏa mãn, trả về `-1`.

| `nums` | `k` | Output | Giải thích |
|---|---:|---:|---|
| `[34,23,1,24,75,33,54,8]` | `60` | `58` | `34 + 24 = 58 < 60`, không cặp nào cho 59 |
| `[10,20,30]` | `15` | `-1` | cặp nhỏ nhất là `10+20 = 30 ≥ 15` |

**Constraints (theo đề gốc):**
- `1 <= nums.length <= 100`
- `1 <= nums[i] <= 1000`
- `1 <= k <= 2000`

Điểm cần để ý ngay:
- Điều kiện là `<` **chặt** (strict), không phải `<=`.
- `i < j` nghĩa là **hai vị trí khác nhau**; một phần tử không được dùng hai lần (nhưng hai phần tử *khác vị trí* có thể bằng giá trị).
- Mảng có thể chỉ có 1 phần tử → không có cặp nào → `-1`.

In [1]:
nums = [34, 23, 1, 24, 75, 33, 54, 8]
k = 60
# Kiểm tra bằng tay vài cặp
print("34 + 24 =", 34 + 24, "<", k, "?", 34 + 24 < k)
print("33 + 24 =", 33 + 24, "<", k, "?", 33 + 24 < k)
print("54 + 8  =", 54 + 8, "<", k, "?", 54 + 8 < k)   # 62 >= 60 -> không hợp lệ

34 + 24 = 58 < 60 ? True
33 + 24 = 57 < 60 ? True
54 + 8  = 62 < 60 ? False


## 2. Nền tảng Python cần dùng

### 2.1. Duyệt mọi cặp `i < j` bằng hai vòng `for` lồng nhau

`range(a, b)` sinh các số `a, a+1, ..., b-1`. Để liệt kê mọi cặp chỉ số **không lặp lại và không trùng vị trí**, vòng ngoài chạy `i`, vòng trong bắt đầu từ `i + 1`.

Với `n = 4`, các cặp là `(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)` — tổng cộng `n*(n-1)/2 = 6` cặp.

In [2]:
n = 4
pairs = []
for i in range(n):
    for j in range(i + 1, n):   # j luôn > i
        pairs.append((i, j))
print(pairs)
print("Số cặp:", len(pairs), "= n*(n-1)//2 =", n * (n - 1) // 2)

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
Số cặp: 6 = n*(n-1)//2 = 6


### 2.2. `max()` và giá trị khởi tạo `-1`

Ta cần "giữ lại giá trị lớn nhất từng thấy". Mẹo: khởi tạo `best = -1` (đúng giá trị phải trả khi không có cặp). Mỗi khi gặp tổng hợp lệ, cập nhật `best = max(best, s)`.

Vì mọi `nums[i] >= 1`, tổng hợp lệ luôn `>= 2 > -1`, nên chỉ cần thấy một cặp hợp lệ là `best` bị thay thế.

In [3]:
best = -1
for s in [30, 58, 12, 61]:
    if s < 60:
        best = max(best, s)
    print("xét", s, "-> best =", best)

xét 30 -> best = 30
xét 58 -> best = 58
xét 12 -> best = 58
xét 61 -> best = 58


### 2.3. `sorted()` và `list.sort()`

- `sorted(a)` **trả về list mới** đã sắp xếp tăng dần, `a` giữ nguyên.
- `a.sort()` sắp xếp **tại chỗ** (in-place), trả về `None`.

Sắp xếp trong Python (Timsort) tốn `O(n log n)`. Sau khi sắp xếp, phần tử nhỏ nhất ở đầu, lớn nhất ở cuối — đây là chìa khóa cho hai con trỏ.

In [4]:
a = [34, 23, 1, 24, 75, 33, 54, 8]
b_sorted = sorted(a)
print("sorted(a):", b_sorted)
print("a vẫn giữ nguyên:", a)
r = a.sort()
print("a.sort() trả về:", r, "| a sau khi sort:", a)

sorted(a): [1, 8, 23, 24, 33, 34, 54, 75]
a vẫn giữ nguyên: [34, 23, 1, 24, 75, 33, 54, 8]
a.sort() trả về: None | a sau khi sort: [1, 8, 23, 24, 33, 34, 54, 75]


## 3. Cách 1 — Brute Force: thử mọi cặp

Ý tưởng trực giác: thử **mọi** cặp `(i, j)` với `i < j`, tính tổng, nếu `< k` thì so với kết quả tốt nhất.

```text
best = -1
với mọi i < j:
    s = nums[i] + nums[j]
    nếu s < k: best = max(best, s)
trả về best
```

In [5]:
from typing import List

def two_sum_less_than_k_brute(nums: List[int], k: int) -> int:
    best = -1
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            s = nums[i] + nums[j]
            if s < k:
                best = max(best, s)
    return best

print(two_sum_less_than_k_brute([34, 23, 1, 24, 75, 33, 54, 8], 60))  # 58
print(two_sum_less_than_k_brute([10, 20, 30], 15))                      # -1
print(two_sum_less_than_k_brute([5], 100))                              # -1 (chỉ 1 phần tử)

58
-1
-1


### Độ phức tạp của Brute Force và nút thắt

- Số cặp: `n(n-1)/2` → Time `O(n²)`, Space `O(1)`.
- Với `n <= 100` thì chỉ ~4950 cặp, nộp vẫn qua. Nhưng bài này là bài **luyện kỹ thuật**: nếu `n = 10^5` thì `5·10^9` cặp → quá chậm.

**Nút thắt:** ta kiểm tra cả những cặp "chắc chắn vô ích". Ví dụ khi đã biết `x + y >= k` với `y` là số lớn nhất, thì không cần thử `y` với số còn lớn hơn `x`. Muốn khai thác kiểu suy luận "lớn hơn / nhỏ hơn" này, ta cần dữ liệu **có thứ tự** → sắp xếp.

In [6]:
import time

def count_pairs_checked(n):
    return n * (n - 1) // 2

for n in [100, 1000, 10**5]:
    print(f"n = {n:>6}: số cặp brute force phải xét = {count_pairs_checked(n):,}")

n =    100: số cặp brute force phải xét = 4,950
n =   1000: số cặp brute force phải xét = 499,500
n = 100000: số cặp brute force phải xét = 4,999,950,000


## 4. Insight — sau khi sắp xếp, mỗi lần so sánh loại được cả một dãy cặp

Sắp xếp `nums` tăng dần, đặt `lo = 0` (số nhỏ nhất), `hi = n-1` (số lớn nhất). Xét `s = nums[lo] + nums[hi]`:

1. **Nếu `s >= k`**: tổng quá lớn. `nums[hi]` ghép với `nums[lo]` (số nhỏ nhất còn lại) mà vẫn quá lớn, thì ghép với **bất kỳ** số nào khác còn lại cũng quá lớn. → `nums[hi]` vô dụng, bỏ nó: `hi -= 1`.
2. **Nếu `s < k`**: cặp hợp lệ, ghi nhận `best = max(best, s)`. Với `nums[lo]`, đây là cặp **tốt nhất có thể** (vì `nums[hi]` là số lớn nhất còn lại). → `nums[lo]` đã "dùng hết tiềm năng", bỏ nó: `lo += 1`.

Mỗi bước loại đúng 1 phần tử → tối đa `n - 1` bước. Cộng chi phí sort → `O(n log n)`.

In [7]:
arr = sorted([34, 23, 1, 24, 75, 33, 54, 8])
print("Sau sắp xếp:", arr)
lo, hi, k = 0, len(arr) - 1, 60
s = arr[lo] + arr[hi]
print(f"Bước đầu: arr[lo]={arr[lo]}, arr[hi]={arr[hi]}, s={s}, s<k? {s < k}")
print("Vì s >= k, mọi cặp (x, 75) với x >= 1 đều >= 76 -> bỏ 75")

Sau sắp xếp: [1, 8, 23, 24, 33, 34, 54, 75]
Bước đầu: arr[lo]=1, arr[hi]=75, s=76, s<k? False
Vì s >= k, mọi cặp (x, 75) với x >= 1 đều >= 76 -> bỏ 75


## 5. Cách 2 — Sắp xếp + Hai con trỏ (lời giải chính)

### 5.1. Từng mảnh ghép

- **Khởi tạo:** `nums` đã sắp xếp; `lo, hi = 0, len(nums) - 1`; `best = -1`.
- **Điều kiện lặp:** `while lo < hi:` — hai con trỏ phải trỏ tới **hai vị trí khác nhau** (không dùng một phần tử hai lần).
- **Thân vòng lặp:** tính `s`, rồi rẽ nhánh như mục 4.
- **Kết thúc:** trả `best`.

In [8]:
# Minh họa riêng phần "điều kiện lo < hi"
arr = [3, 8]
lo, hi = 0, 1
print("lo < hi ?", lo < hi, "-> còn một cặp để xét:", (arr[lo], arr[hi]))
lo += 1
print("sau lo += 1: lo < hi ?", lo < hi, "-> dừng, không được ghép arr[1] với chính nó")

lo < hi ? True -> còn một cặp để xét: (3, 8)
sau lo += 1: lo < hi ? False -> dừng, không được ghép arr[1] với chính nó


In [9]:
def two_sum_less_than_k_two_pointers(nums: List[int], k: int) -> int:
    arr = sorted(nums)          # không sửa input gốc
    lo, hi = 0, len(arr) - 1
    best = -1
    while lo < hi:
        s = arr[lo] + arr[hi]
        if s < k:
            best = max(best, s)  # cặp hợp lệ tốt nhất cho arr[lo]
            lo += 1              # thử số nhỏ lớn hơn để tổng tăng lên
        else:
            hi -= 1              # tổng quá lớn -> bỏ số lớn nhất
    return best

print(two_sum_less_than_k_two_pointers([34, 23, 1, 24, 75, 33, 54, 8], 60))  # 58
print(two_sum_less_than_k_two_pointers([10, 20, 30], 15))                      # -1
print(two_sum_less_than_k_two_pointers([1, 1], 3))                            # 2

58
-1
2


### 5.2. Tại sao khi `s < k` lại tăng `lo` mà không giảm `hi`?

Khi `s < k`, ta muốn tổng **lớn hơn nữa** (gần `k` hơn). Giảm `hi` chỉ làm tổng nhỏ đi → vô ích. Tăng `lo` cho số nhỏ lớn hơn → tổng có thể tăng.

Ngoài ra `arr[lo]` không thể có cặp nào tốt hơn `arr[lo] + arr[hi]` (vì các số `> arr[hi]` đã bị loại do quá lớn). Nên bỏ `arr[lo]` là an toàn.

## 6. TRACE từng bước

Ví dụ `nums = [34,23,1,24,75,33,54,8]`, `k = 60` → sau sort `[1, 8, 23, 24, 33, 34, 54, 75]`.

| Bước | lo | hi | arr[lo] | arr[hi] | s | s < k? | Hành động | best |
|---:|---:|---:|---:|---:|---:|:---:|---|---:|
| 1 | 0 | 7 | 1 | 75 | 76 | Không | `hi -= 1` | -1 |
| 2 | 0 | 6 | 1 | 54 | 55 | Có | best=55, `lo += 1` | 55 |
| 3 | 1 | 6 | 8 | 54 | 62 | Không | `hi -= 1` | 55 |
| 4 | 1 | 5 | 8 | 34 | 42 | Có | best=55, `lo += 1` | 55 |
| 5 | 2 | 5 | 23 | 34 | 57 | Có | best=57, `lo += 1` | 57 |
| 6 | 3 | 5 | 24 | 34 | 58 | Có | best=58, `lo += 1` | 58 |
| 7 | 4 | 5 | 33 | 34 | 67 | Không | `hi -= 1` | 58 |
| — | 4 | 4 | | | | | dừng vì `lo == hi` | **58** |

Ý nghĩa các cột: `lo/hi` là chỉ số hai con trỏ **trước** khi cập nhật; `best` là giá trị **sau** bước đó.

In [10]:
def trace_two_sum_less_than_k(nums: List[int], k: int) -> int:
    arr = sorted(nums)
    print("arr sau sort:", arr, "| k =", k)
    lo, hi = 0, len(arr) - 1
    best = -1
    step = 1
    while lo < hi:
        s = arr[lo] + arr[hi]
        if s < k:
            best = max(best, s)
            action = "hợp lệ -> lo += 1"
            lo_new, hi_new = lo + 1, hi
        else:
            action = "quá lớn -> hi -= 1"
            lo_new, hi_new = lo, hi - 1
        print(f"Bước {step}: lo={lo}, hi={hi}, {arr[lo]}+{arr[hi]}={s:>3} | {action:<18} | best={best}")
        lo, hi = lo_new, hi_new
        step += 1
    print("Dừng: lo =", lo, ", hi =", hi, "-> kết quả", best)
    return best

_ = trace_two_sum_less_than_k([34, 23, 1, 24, 75, 33, 54, 8], 60)

arr sau sort: [1, 8, 23, 24, 33, 34, 54, 75] | k = 60
Bước 1: lo=0, hi=7, 1+75= 76 | quá lớn -> hi -= 1 | best=-1
Bước 2: lo=0, hi=6, 1+54= 55 | hợp lệ -> lo += 1  | best=55
Bước 3: lo=1, hi=6, 8+54= 62 | quá lớn -> hi -= 1 | best=55
Bước 4: lo=1, hi=5, 8+34= 42 | hợp lệ -> lo += 1  | best=55
Bước 5: lo=2, hi=5, 23+34= 57 | hợp lệ -> lo += 1  | best=57
Bước 6: lo=3, hi=5, 24+34= 58 | hợp lệ -> lo += 1  | best=58
Bước 7: lo=4, hi=5, 33+34= 67 | quá lớn -> hi -= 1 | best=58
Dừng: lo = 4 , hi = 4 -> kết quả 58


Trace ví dụ không có đáp án: `nums = [10, 20, 30]`, `k = 15`. Mọi bước đều rơi vào nhánh "quá lớn", `best` giữ nguyên `-1`.

In [11]:
_ = trace_two_sum_less_than_k([10, 20, 30], 15)

arr sau sort: [10, 20, 30] | k = 15
Bước 1: lo=0, hi=2, 10+30= 40 | quá lớn -> hi -= 1 | best=-1
Bước 2: lo=0, hi=1, 10+20= 30 | quá lớn -> hi -= 1 | best=-1
Dừng: lo = 0 , hi = 0 -> kết quả -1


## 7. Code nộp LeetCode — Hai con trỏ

Lưu ý LeetCode dùng `List` từ `typing` (trên LeetCode đã import sẵn; trong notebook ta tự import).

In [12]:
from typing import List

class Solution:
    def twoSumLessThanK(self, nums: List[int], k: int) -> int:
        nums.sort()
        lo, hi = 0, len(nums) - 1
        best = -1
        while lo < hi:
            s = nums[lo] + nums[hi]
            if s < k:
                best = max(best, s)
                lo += 1
            else:
                hi -= 1
        return best

sol = Solution()
print(sol.twoSumLessThanK([34, 23, 1, 24, 75, 33, 54, 8], 60))  # 58
print(sol.twoSumLessThanK([10, 20, 30], 15))                      # -1

58
-1


## 8. Vì sao hai con trỏ đúng? (invariant)

**Invariant (bất biến):** tại mỗi thời điểm, mọi cặp `(i, j)` có `i < lo` hoặc `j > hi` **đều đã được xử lý**: hoặc đã được tính vào `best`, hoặc chắc chắn không tốt hơn `best`/không hợp lệ.

- Khi bỏ `hi` (vì `arr[lo] + arr[hi] >= k`): với mọi `i` trong `[lo, hi)`, `arr[i] >= arr[lo]` nên `arr[i] + arr[hi] >= k` → mọi cặp chứa `hi` đều không hợp lệ.
- Khi bỏ `lo` (vì `arr[lo] + arr[hi] < k`): với mọi `j` trong `(lo, hi]`, `arr[j] <= arr[hi]` nên `arr[lo] + arr[j] <= arr[lo] + arr[hi]` → cặp `(lo, hi)` là cặp tốt nhất chứa `lo`, và nó đã vào `best`.

Khi `lo == hi`, không còn cặp nào chưa xử lý → `best` là đáp án. Không bao giờ ghép một phần tử với chính nó vì luôn có `lo < hi`.

## 9. Độ phức tạp

Gọi `n = len(nums)`.

| Phần | Time | Space |
|---|---|---|
| Sắp xếp | `O(n log n)` | `O(n)` nếu dùng `sorted` (tạo bản sao); `nums.sort()` tại chỗ (Timsort vẫn cần bộ nhớ phụ tối đa `O(n)` trong worst case) |
| Hai con trỏ | `O(n)` — mỗi bước loại 1 phần tử | `O(1)` |
| **Tổng** | `O(n log n)` | `O(1)` ngoài phần sắp xếp |

## 10. Cách 3 — Sắp xếp + Tìm nhị phân (`bisect`)

Ý tưởng khác: với mỗi `nums[i]` (đã sort), ta muốn số `y` **lớn nhất** đứng **sau** `i` sao cho `nums[i] + y < k`, tức `y < k - nums[i]` (hay `y <= k - nums[i] - 1`).

### 10.1. `bisect_left` là gì?

`bisect.bisect_left(a, x, lo, hi)` trả về vị trí đầu tiên trong `a[lo:hi]` (đã sort) mà `a[pos] >= x`. Nói cách khác: số phần tử `< x` nằm ở `a[lo:pos]`. Vậy phần tử lớn nhất `< x` là `a[pos - 1]` (nếu `pos - 1 >= lo`).

In [13]:
import bisect

a = [1, 8, 23, 24, 33, 34, 54, 75]
for x in [24, 25, 1, 100]:
    pos = bisect.bisect_left(a, x)
    print(f"x={x:>3}: bisect_left -> {pos}; phần tử lớn nhất < x là",
          a[pos - 1] if pos > 0 else "không có")
# Giới hạn vùng tìm bằng lo, hi
print("Chỉ tìm trong a[3:]:", bisect.bisect_left(a, 30, 3))

x= 24: bisect_left -> 3; phần tử lớn nhất < x là 23
x= 25: bisect_left -> 4; phần tử lớn nhất < x là 24
x=  1: bisect_left -> 0; phần tử lớn nhất < x là không có
x=100: bisect_left -> 8; phần tử lớn nhất < x là 75
Chỉ tìm trong a[3:]: 4


In [14]:
def two_sum_less_than_k_binary_search(nums: List[int], k: int) -> int:
    arr = sorted(nums)
    n = len(arr)
    best = -1
    for i in range(n):
        need_less_than = k - arr[i]                 # cần y < need_less_than
        pos = bisect.bisect_left(arr, need_less_than, i + 1, n)
        j = pos - 1                                 # ứng viên lớn nhất < need_less_than
        if j > i:                                   # phải là vị trí sau i
            best = max(best, arr[i] + arr[j])
    return best

print(two_sum_less_than_k_binary_search([34, 23, 1, 24, 75, 33, 54, 8], 60))  # 58
print(two_sum_less_than_k_binary_search([10, 20, 30], 15))                      # -1

58
-1


Code nộp LeetCode cho cách 3 (Time `O(n log n)`: sort + `n` lần tìm nhị phân `O(log n)`; Space `O(1)` ngoài sort):

In [15]:
class SolutionBinarySearch:
    def twoSumLessThanK(self, nums: List[int], k: int) -> int:
        nums.sort()
        best = -1
        for i, x in enumerate(nums):
            j = bisect.bisect_left(nums, k - x, i + 1) - 1
            if j > i:
                best = max(best, x + nums[j])
        return best

print(SolutionBinarySearch().twoSumLessThanK([34, 23, 1, 24, 75, 33, 54, 8], 60))

58


## 11. Cách 4 — Đếm tần suất (Counting Sort) tận dụng `nums[i] <= 1000`

Vì giá trị chỉ từ `1` tới `1000`, ta có thể "sắp xếp" bằng cách **đếm**: `count[v]` = số lần giá trị `v` xuất hiện. Sau đó chạy hai con trỏ trên **giá trị** `lo = 1`, `hi = 1000` thay vì trên chỉ số.

Điểm khó: khi `lo == hi`, cặp `(v, v)` chỉ hợp lệ nếu `count[v] >= 2` (cần hai phần tử khác vị trí).

Time `O(n + m)` với `m = 1000` (miền giá trị), Space `O(m)`.

In [16]:
def two_sum_less_than_k_counting(nums: List[int], k: int) -> int:
    MAXV = 1000
    count = [0] * (MAXV + 1)
    for x in nums:
        count[x] += 1
    best = -1
    lo, hi = 1, MAXV
    while lo <= hi:
        if count[lo] == 0:            # giá trị lo không có trong mảng
            lo += 1
            continue
        if count[hi] == 0:
            hi -= 1
            continue
        if lo == hi and count[lo] < 2:  # không đủ 2 phần tử cùng giá trị
            break
        s = lo + hi
        if s < k:
            best = max(best, s)
            lo += 1
        else:
            hi -= 1
    return best

print(two_sum_less_than_k_counting([34, 23, 1, 24, 75, 33, 54, 8], 60))  # 58
print(two_sum_less_than_k_counting([10, 20, 30], 15))                      # -1
print(two_sum_less_than_k_counting([5, 5], 11))                            # 10 (dùng 2 số 5)
print(two_sum_less_than_k_counting([5], 11))                               # -1

58
-1
10
-1


### Vì sao `lo == hi` vẫn xét được?

Ở cách hai con trỏ trên chỉ số, `lo < hi` đảm bảo 2 vị trí khác nhau. Ở đây con trỏ chạy trên **giá trị**: một giá trị có thể có nhiều bản sao. Nên `lo == hi` hợp lệ khi `count[lo] >= 2`. Đây là chi tiết dễ quên khi chuyển từ chỉ số sang giá trị.

### 11.1. So sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute Force | thử mọi cặp | `O(n²)` | `O(1)` | `n` nhỏ, để kiểm chứng |
| 2. Sort + hai con trỏ | loại 1 phần tử mỗi bước | `O(n log n)` | `O(1)`* | Lời giải chuẩn, nên nhớ |
| 3. Sort + bisect | tìm số lớn nhất `< k - x` | `O(n log n)` | `O(1)`* | Khi quen tìm nhị phân |
| 4. Đếm + hai con trỏ trên giá trị | counting sort | `O(n + m)` | `O(m)` | Miền giá trị nhỏ (`m = 1000`) |

\* không tính bộ nhớ của thao tác sắp xếp.

## 12. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ví dụ của đề.
2. Biên: 1 phần tử; 2 phần tử; `k` rất nhỏ (`k = 1`) → luôn `-1`.
3. Dễ sai: tổng **bằng đúng** `k` (phải loại); hai phần tử cùng giá trị; nhiều bản sao.

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<32} PASS ({actual})")

tests = [
    ("mẫu 1", [34, 23, 1, 24, 75, 33, 54, 8], 60, 58),
    ("mẫu 2", [10, 20, 30], 15, -1),
    ("1 phần tử", [7], 100, -1),
    ("2 phần tử hợp lệ", [3, 4], 8, 7),
    ("tổng bằng đúng k -> loại", [3, 4], 7, -1),
    ("k = 1", [1, 1, 1], 1, -1),
    ("hai số bằng nhau", [5, 5, 9], 11, 10),
    ("nhiều bản sao", [1, 1, 1, 1], 3, 2),
    ("giá trị lớn", [1000, 999, 1], 2000, 1999),
]
for name, nums, k, expected in tests:
    check(name, Solution().twoSumLessThanK(list(nums), k), expected)

mẫu 1                            PASS (58)
mẫu 2                            PASS (-1)
1 phần tử                        PASS (-1)
2 phần tử hợp lệ                 PASS (7)
tổng bằng đúng k -> loại         PASS (-1)
k = 1                            PASS (-1)
hai số bằng nhau                 PASS (10)
nhiều bản sao                    PASS (2)
giá trị lớn                      PASS (1999)


### Kiểm thử đối chiếu ngẫu nhiên có seed

Dùng Brute Force làm "oracle" (đáp án chắc chắn đúng), so sánh với 3 cách còn lại trên nhiều input ngẫu nhiên. `random.Random(seed)` tạo bộ sinh số ngẫu nhiên lặp lại được.

In [18]:
import random

rng = random.Random(1099)
for _ in range(2000):
    n = rng.randint(1, 12)
    nums = [rng.randint(1, 30) for _ in range(n)]
    k = rng.randint(1, 65)
    expected = two_sum_less_than_k_brute(nums, k)
    assert two_sum_less_than_k_two_pointers(nums, k) == expected, (nums, k)
    assert two_sum_less_than_k_binary_search(nums, k) == expected, (nums, k)
    assert two_sum_less_than_k_counting(nums, k) == expected, (nums, k)
    assert Solution().twoSumLessThanK(list(nums), k) == expected, (nums, k)
print("2000 test ngẫu nhiên: 4 cách đều khớp Brute Force")

2000 test ngẫu nhiên: 4 cách đều khớp Brute Force


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `<=` thay vì `<`

In [19]:
def wrong_le(nums, k):
    arr = sorted(nums); lo, hi = 0, len(arr) - 1; best = -1
    while lo < hi:
        s = arr[lo] + arr[hi]
        if s <= k:            # SAI: cho phép tổng bằng k
            best = max(best, s); lo += 1
        else:
            hi -= 1
    return best

print("sai :", wrong_le([3, 4], 7), "| đúng:", two_sum_less_than_k_two_pointers([3, 4], 7))

sai : 7 | đúng: -1


**Vì sao sai:** đề yêu cầu `sum < k` (nhỏ hơn hẳn). Với `[3,4], k=7`, tổng `7` không hợp lệ → đáp án `-1`. **Sửa:** dùng `s < k`.

### Lỗi 2: Dùng `while lo <= hi` → ghép một phần tử với chính nó

In [20]:
def wrong_same_element(nums, k):
    arr = sorted(nums); lo, hi = 0, len(arr) - 1; best = -1
    while lo <= hi:           # SAI: cho phép lo == hi
        s = arr[lo] + arr[hi]
        if s < k:
            best = max(best, s); lo += 1
        else:
            hi -= 1
    return best

print("sai :", wrong_same_element([5], 11), "| đúng:", two_sum_less_than_k_two_pointers([5], 11))

sai : 10 | đúng: -1


**Vì sao sai:** khi `lo == hi`, `arr[lo] + arr[hi]` là **một** phần tử cộng chính nó (`5 + 5 = 10`), vi phạm `i < j`. **Sửa:** `while lo < hi`.

### Lỗi 3: Quên sắp xếp trước khi dùng hai con trỏ

In [21]:
def wrong_no_sort(nums, k):
    lo, hi = 0, len(nums) - 1; best = -1
    while lo < hi:
        s = nums[lo] + nums[hi]
        if s < k:
            best = max(best, s); lo += 1
        else:
            hi -= 1
    return best

nums = [34, 23, 1, 24, 75, 33, 54, 8]
print("không sort:", wrong_no_sort(nums, 60), "| đúng:", two_sum_less_than_k_two_pointers(nums, 60))

không sort: 42 | đúng: 58


**Vì sao sai:** lập luận "bỏ `hi` vì nó ghép với số nhỏ nhất mà vẫn quá lớn" chỉ đúng khi `arr[lo]` thật sự nhỏ nhất và `arr[hi]` lớn nhất — tức mảng phải **đã sắp xếp**. **Sửa:** `nums.sort()` trước.

### Lỗi 4: Khi `s < k` lại giảm `hi`

Giảm `hi` làm tổng nhỏ đi, ta đi xa khỏi `k` và bỏ mất các cặp tốt. Ví dụ `[1, 8, 23, 24, 33, 34]`, `k = 60`: bước đầu `1 + 34 = 35 < 60`, nếu giảm `hi` thì `34` bị loại sớm và không còn tìm được `24 + 34 = 58`.

In [22]:
def wrong_move_hi(nums, k):
    arr = sorted(nums); lo, hi = 0, len(arr) - 1; best = -1
    while lo < hi:
        s = arr[lo] + arr[hi]
        if s < k:
            best = max(best, s); hi -= 1   # SAI hướng
        else:
            hi -= 1
    return best

print("sai :", wrong_move_hi([1, 8, 23, 24, 33, 34], 60), "| đúng:", two_sum_less_than_k_two_pointers([1, 8, 23, 24, 33, 34], 60))

sai : 35 | đúng: 58


## 14. Mẹo nhận dạng pattern

- Đề hỏi về **cặp** phần tử và **tổng so với một ngưỡng** (`<`, `<=`, `==`, gần nhất) → nghĩ ngay **sort + hai con trỏ**.
- Không cần trả **chỉ số gốc** (chỉ cần giá trị tổng) → sắp xếp được thoải mái.
- Nếu đề cần chỉ số gốc (như Two Sum #1) → dùng hash map, hoặc sort kèm chỉ số.
- Miền giá trị nhỏ (`<= 1000`) → cân nhắc counting sort.
- Bài cùng họ: #1 Two Sum, #167 Two Sum II, #15 3Sum, #16 3Sum Closest, #259 3Sum Smaller, #881 Boats to Save People.

## 15. Biến thể

### 15.1. Đếm số cặp có tổng `< k` (giống #259 nhưng cho 2 số)

Khi `arr[lo] + arr[hi] < k`, thì `arr[lo]` ghép với **mọi** `j` trong `(lo, hi]` đều `< k` → cộng `hi - lo` cặp một lần.

In [23]:
def count_pairs_less_than_k(nums: List[int], k: int) -> int:
    arr = sorted(nums)
    lo, hi = 0, len(arr) - 1
    cnt = 0
    while lo < hi:
        if arr[lo] + arr[hi] < k:
            cnt += hi - lo      # (lo, lo+1), ..., (lo, hi) đều hợp lệ
            lo += 1
        else:
            hi -= 1
    return cnt

print(count_pairs_less_than_k([1, 2, 3, 4], 6))  # cặp: (1,2)(1,3)(1,4)(2,3) = 4

4


### 15.2. Tổng gần `k` nhất (không quan tâm lớn hơn hay nhỏ hơn)

Giữ `best` có `|s - k|` nhỏ nhất; nếu `s == k` thì trả ngay.

In [24]:
def two_sum_closest(nums: List[int], k: int) -> int:
    arr = sorted(nums)
    lo, hi = 0, len(arr) - 1
    best = arr[0] + arr[1]
    while lo < hi:
        s = arr[lo] + arr[hi]
        if abs(s - k) < abs(best - k):
            best = s
        if s == k:
            return s
        if s < k:
            lo += 1
        else:
            hi -= 1
    return best

print(two_sum_closest([34, 23, 1, 24, 75, 33, 54, 8], 60))  # 62 (cách 60 hai đơn vị)
print(two_sum_closest([10, 20, 30], 15))                      # 30

62
30


Ở ví dụ đầu, `1 + 54 = 55`, `8 + 54 = 62`, `24 + 34 = 58`, `23 + 34 = 57`… tổng gần 60 nhất là `58` hoặc `62` (cùng cách 2). Hàm chỉ thay `best` khi khoảng cách **nhỏ hơn hẳn**, nên khi hòa nó giữ giá trị **gặp trước**: `8 + 54 = 62` được gặp trước `24 + 34 = 58` → trả `62`. Đây cũng là bài học: với "gần nhất", phải nói rõ quy tắc hòa.

## 16. Bài tập (tăng dần)

**Bài 1 — Tính tay:** Với `nums = [5, 1, 9, 3, 7]`, `k = 11`, hãy sort và lập bảng trace hai con trỏ. Đáp án là bao nhiêu?

**Bài 2 — Dự đoán output:** `Solution().twoSumLessThanK([6, 6, 6], 12)` trả gì? Còn với `k = 13`?

**Bài 3 — TODO:** hoàn thiện hai con trỏ.

In [25]:
def exercise_two_sum_less_than_k(nums: List[int], k: int) -> int:
    # TODO 1: sắp xếp
    # TODO 2: lo, hi, best = 0, len - 1, -1
    # TODO 3: while lo < hi: tính s, rẽ nhánh
    pass

# assert exercise_two_sum_less_than_k([34, 23, 1, 24, 75, 33, 54, 8], 60) == 58
# assert exercise_two_sum_less_than_k([10, 20, 30], 15) == -1

**Bài 4 — Biến thể:** trả về **cặp giá trị** `(a, b)` với `a <= b` đạt tổng lớn nhất `< k`, hoặc `None` nếu không có.

In [26]:
def exercise_best_pair(nums: List[int], k: int):
    # TODO: giống hai con trỏ nhưng lưu thêm cặp
    pass

# assert exercise_best_pair([34, 23, 1, 24, 75, 33, 54, 8], 60) == (24, 34)
# assert exercise_best_pair([10, 20, 30], 15) is None

**Bài 5 — Đếm cặp `<= k`:** đếm số cặp `i < j` có `nums[i] + nums[j] <= k`.

In [27]:
def exercise_count_pairs_at_most_k(nums: List[int], k: int) -> int:
    # TODO
    pass

# assert exercise_count_pairs_at_most_k([1, 2, 3, 4], 5) == 4

**Bài 6 — Mini challenge (3 số):** tìm tổng lớn nhất của **ba** phần tử khác vị trí sao cho tổng `< k`, trả `-1` nếu không có. Gợi ý: cố định một số rồi dùng hai con trỏ cho phần còn lại → `O(n²)`.

In [28]:
def exercise_three_sum_less_than_k(nums: List[int], k: int) -> int:
    # TODO
    pass

# assert exercise_three_sum_less_than_k([1, 2, 3, 4, 5], 10) == 9
# assert exercise_three_sum_less_than_k([5, 5, 5], 15) == -1

### 16.1. Gợi ý 3 mức

| Bài | Mức 1 (hướng nghĩ) | Mức 2 (cấu trúc) | Mức 3 (gần code) |
|---|---|---|---|
| 1 | Sort trước | `[1,3,5,7,9]` | `1+9=10<11` → best=10, lo++ ; `3+9=12` → hi-- ... |
| 2 | Nhớ `<` chặt | Mọi cặp đều = 12 | `k=12` → `-1`; `k=13` → `12` |
| 3 | Làm như mục 5 | `while lo < hi` | `if s < k: best=max(best,s); lo+=1 else: hi-=1` |
| 4 | Lưu cả cặp khi cập nhật best | `best_pair = None` | `if s < k and s > best: best, pair = s, (arr[lo], arr[hi])` |
| 5 | Giống biến thể 15.1 | đổi `<` thành `<=` | `cnt += hi - lo` |
| 6 | Cố định `i` | `lo = i+1`, `hi = n-1` | lặp hai con trỏ trong vòng `for i` |

## 17. Đáp án tham khảo (chỉ xem sau khi tự làm)

- **Bài 1:** sort → `[1,3,5,7,9]`. `1+9=10<11` (best=10, lo=1) → `3+9=12` (hi=3) → `3+7=10<11` (lo=2) → `5+7=12` (hi=2) → dừng. Đáp án **10**.
- **Bài 2:** `k=12` → `-1` (mọi tổng = 12, không `< 12`); `k=13` → `12`.

In [29]:
def answer_two_sum_less_than_k(nums, k):
    arr = sorted(nums); lo, hi = 0, len(arr) - 1; best = -1
    while lo < hi:
        s = arr[lo] + arr[hi]
        if s < k:
            best = max(best, s); lo += 1
        else:
            hi -= 1
    return best

def answer_best_pair(nums, k):
    arr = sorted(nums); lo, hi = 0, len(arr) - 1
    best, pair = -1, None
    while lo < hi:
        s = arr[lo] + arr[hi]
        if s < k:
            if s > best:
                best, pair = s, (arr[lo], arr[hi])
            lo += 1
        else:
            hi -= 1
    return pair

def answer_count_pairs_at_most_k(nums, k):
    arr = sorted(nums); lo, hi = 0, len(arr) - 1; cnt = 0
    while lo < hi:
        if arr[lo] + arr[hi] <= k:
            cnt += hi - lo; lo += 1
        else:
            hi -= 1
    return cnt

def answer_three_sum_less_than_k(nums, k):
    arr = sorted(nums); n = len(arr); best = -1
    for i in range(n - 2):
        lo, hi = i + 1, n - 1
        while lo < hi:
            s = arr[i] + arr[lo] + arr[hi]
            if s < k:
                best = max(best, s); lo += 1
            else:
                hi -= 1
    return best

assert answer_two_sum_less_than_k([5, 1, 9, 3, 7], 11) == 10
assert answer_two_sum_less_than_k([6, 6, 6], 12) == -1
assert answer_two_sum_less_than_k([6, 6, 6], 13) == 12
assert answer_two_sum_less_than_k([34, 23, 1, 24, 75, 33, 54, 8], 60) == 58
assert answer_best_pair([34, 23, 1, 24, 75, 33, 54, 8], 60) == (24, 34)
assert answer_best_pair([10, 20, 30], 15) is None
assert answer_count_pairs_at_most_k([1, 2, 3, 4], 5) == 4
assert answer_three_sum_less_than_k([1, 2, 3, 4, 5], 10) == 9
assert answer_three_sum_less_than_k([5, 5, 5], 15) == -1

# Đối chiếu đáp án bài 5, 6 với brute force
from itertools import combinations
rng = random.Random(7)
for _ in range(500):
    nums = [rng.randint(1, 20) for _ in range(rng.randint(1, 9))]
    k = rng.randint(1, 50)
    assert answer_count_pairs_at_most_k(nums, k) == sum(1 for a, b2 in combinations(nums, 2) if a + b2 <= k)
    sums3 = [sum(c) for c in combinations(nums, 3) if sum(c) < k]
    assert answer_three_sum_less_than_k(nums, k) == (max(sums3) if sums3 else -1)
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


## 18. Template nhớ nhanh + checklist submit

```python
nums.sort()
lo, hi, best = 0, len(nums) - 1, -1
while lo < hi:
    s = nums[lo] + nums[hi]
    if s < k:              # hợp lệ -> ghi nhận, cần tổng lớn hơn
        best = max(best, s)
        lo += 1
    else:                  # quá lớn -> bỏ số lớn nhất
        hi -= 1
return best
```

**Checklist trước khi submit:**
- [ ] Đã sắp xếp mảng?
- [ ] Điều kiện là `s < k` (chặt), không phải `<=`?
- [ ] `while lo < hi` (không ghép một phần tử với chính nó)?
- [ ] `best` khởi tạo `-1` và trả `-1` khi không có cặp?
- [ ] Mảng 1 phần tử chạy đúng?
- [ ] Hướng di chuyển: hợp lệ → `lo += 1`; quá lớn → `hi -= 1`?